In [0]:
#CONFIGURAÇÃO
from pyspark.sql.functions import col, when, lower

# 1. Definição dos caminhos - usando Unity Catalog
BRONZE_TABLE = "breweries_catalog.bronze.breweries_bronze"
SILVER_TABLE = "breweries_catalog.silver.breweries_partitioned"

def transform_bronze_to_silver():
    # Lendo os dados da camada Bronze
    df_bronze = spark.read.table(BRONZE_TABLE)

    # 2. Transformações de Qualidade e Padronização [cite: 13, 14]
    # - Cast de tipos para garantir integridade (ex: coordenadas como float)
    # - Padronização de strings para evitar duplicidade por 'Case Sensitive'
    df_silver = df_bronze.select(
        col("id").cast("string"),
        col("name").cast("string"),
        col("brewery_type").cast("string"),
        col("street").cast("string"),
        col("city").cast("string"),
        # Algumas versões da API usam 'state', outras 'state_province'
        # Aqui garantimos que pegamos a coluna correta
        col("state_province").alias("state"), 
        col("postal_code").cast("string"),
        col("country").cast("string"),
        col("longitude").cast("float"),
        col("latitude").cast("float")
    ).filter(col("id").isNotNull()) # Data Quality: Removendo registros sem ID

    # 3. Escrita com Particionamento 
    # O case pede explicitamente o particionamento por localização.
    # Escolhi 'state' (e opcionalmente 'city') para otimizar filtros regionais.
    df_silver.write.format("delta") \
        .mode("overwrite") \
        .partitionBy("state") \
        .saveAsTable(SILVER_TABLE)

    print(f"Camada Silver atualizada com sucesso e particionada por Estado.")

# Executa a transformação
transform_bronze_to_silver()